In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

In [202]:
raw_data = loadmat("./data/P_data.mat")
raw_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN, Created on: Mon Jan 26 17:25:22 2026',
 '__version__': '1.0',
 '__globals__': [],
 'P_data': array([[1925., 1301., 1320., ..., 2123., 2606., 1318.],
        [1922., 1296., 1320., ..., 2126., 2601., 1317.],
        [1921., 1285., 1316., ..., 2121., 2607., 1316.],
        ...,
        [1253., 1315., 1518., ..., 2615., 1321., 1446.],
        [1248., 1315., 1506., ..., 2613., 1320., 1469.],
        [1249., 1317., 1516., ..., 2605., 1320., 1463.]], shape=(86400, 31)),
 'day_data': array([[array(['2025/11/28'], dtype='<U10'),
         array(['2025/11/29'], dtype='<U10'),
         array(['2025/11/30'], dtype='<U10'),
         array(['2025/12/01'], dtype='<U10'),
         array(['2025/12/02'], dtype='<U10'),
         array(['2025/12/03'], dtype='<U10'),
         array(['2025/12/04'], dtype='<U10'),
         array(['2025/12/05'], dtype='<U10'),
         array(['2025/12/06'], dtype='<U10'),
         array(['2025/12/07'], dtype='<U10'),
     

In [212]:
dates = pd.to_datetime([d[0] for d in raw_data["day_data"].squeeze()])
day_classifications = [d[0] for d in raw_data["day_class"].squeeze()]

df = (
    pd.DataFrame(raw_data["P_data"], columns=dates)
    .assign(second_of_day=np.arange(86400))
    .melt(id_vars="second_of_day", var_name="date", value_name="load")
    .assign(
        date=lambda x: pd.to_datetime(x["date"]),
        timestamp=lambda x: x["date"]
        + pd.to_timedelta(x["second_of_day"], unit="s")
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

day_class_df = pd.DataFrame({"date": dates, "day_class": day_classifications})
df["date"] = pd.to_datetime(df["timestamp"].dt.date)
df = df.merge(day_class_df, on="date", how="left")
df = df[["timestamp", "load", "day_class"]]
print(df.shape)
df.head()

(2678400, 3)


,timestamp,load,day_class
0,2025-11-28 00:00:00,1925.0,full
1,2025-11-28 00:00:01,1922.0,full
2,2025-11-28 00:00:02,1921.0,full
3,2025-11-28 00:00:03,1921.0,full
4,2025-11-28 00:00:04,1926.0,full


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp")
df = df["load"].resample("1min").mean()
df.rename("load", inplace=True)
df = df.reset_index()

df

,timestamp,load
0,2025-11-28 00:00:00,1912.400000
1,2025-11-28 00:01:00,1908.233333
2,2025-11-28 00:02:00,1643.183333
3,2025-11-28 00:03:00,2992.950000
4,2025-11-28 00:04:00,1106.000000
...,...,...
44635,2025-12-28 23:55:00,1443.716667
44636,2025-12-28 23:56:00,1455.533333
44637,2025-12-28 23:57:00,1456.200000
44638,2025-12-28 23:58:00,1459.366667


In [170]:
# ---------------------------------------------------------------
# Resample By Resolution
# ---------------------------------------------------------------
def resample_data(df: pd.DataFrame, resolution: str) -> pd.DataFrame:
    """Resample to target resolution by averaging load per interval."""
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df["load"].resample(resolution).mean()
    df = df.reset_index()
    return df

# ---------------------------------------------------------------
# 1. Temporal Features
# ---------------------------------------------------------------
def time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df["second"] = df["timestamp"].dt.second
    df["minute"] = df["timestamp"].dt.minute
    df["hour"] = df["timestamp"].dt.hour
    df["time_of_day"] = df["hour"].apply(time_of_day)
    df["day"] = df["timestamp"].dt.day
    df["weekday"] = df["timestamp"].dt.dayofweek
    df["is_weekend"] = df["timestamp"].dt.dayofweek.isin([5, 6]).astype(int)
    df["month"] = df["timestamp"].dt.month
    return df

# ---------------------------------------------------------------
# 2. Business Features
# ---------------------------------------------------------------
day_class_map = clean_df.groupby(clean_df["timestamp"].dt.date)["day_class"].first()

def add_business_features(df: pd.DataFrame) -> pd.DataFrame:
    df["workday"] = df["timestamp"].dt.date.map(day_class_map)
    return df

# ---------------------------------------------------------------
# 3. Historic Load Features
# ---------------------------------------------------------------
def add_lag_features(df: pd.DataFrame, intervals: list) -> pd.DataFrame:
    for x in intervals:
        df[f"lag_{x}"] = df["load"].shift(x)
    return df

def add_rolling_features(df: pd.DataFrame, windows: list) -> pd.DataFrame:
    for x in windows:
        df[f"rolling_mean_{x}"] = df["load"].shift(1).rolling(x).mean()
        df[f"rolling_std_{x}"]  = df["load"].shift(1).rolling(x).std()
        df[f"rolling_max_{x}"]  = df["load"].shift(1).rolling(x).max()
        df[f"rolling_min_{x}"]  = df["load"].shift(1).rolling(x).min()
    return df

def add_delta_features(df: pd.DataFrame, intervals: list) -> pd.DataFrame:
    for x in intervals:
        df[f"delta_{x}"] = df["load"].shift(1) - df["load"].shift(x)
    return df

def add_slope_features(df: pd.DataFrame, intervals: list) -> pd.DataFrame:
    def _slope(arr):
        if np.isnan(arr).any():
            return np.nan
        x = np.arange(len(arr))
        return np.polyfit(x, arr, 1)[0]

    for x in intervals:
        df[f"slope_{x}"] = (
            df["load"].shift(1).rolling(x).apply(_slope, raw=True)
        )
    return df

def add_historic_load_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_lag_features(df, LAG_INTERVALS)
    df = add_rolling_features(df, WINDOW_INTERVALS)
    df = add_delta_features(df, WINDOW_INTERVALS)
    df = add_slope_features(df, WINDOW_INTERVALS)

    return df

# ---------------------------------------------------------------
# Resample and Engineer Features by Resolution
# ---------------------------------------------------------------
dfs = {}

print("Resolution Datasets:")
for res in RESAMPLE_RESOLUTIONS:
    print(res, ":", end=" ")
    df = resample_data(clean_df, res)
    df = add_business_features(df)
    df = add_historic_load_features(df)
    df = add_temporal_features(df)
    dfs[res] = df
    print(df.shape)

Resolution Datasets:
30s : (89280, 44)
1min : (44640, 44)
2min : (22320, 44)
5min : (8928, 44)
10min : (4464, 44)
15min : (2976, 44)


In [171]:
dfs["1min"].head()

,timestamp,load,workday,lag_1,lag_2,lag_3,lag_4,lag_5,lag_10,lag_15,...,slope_30,slope_60,second,minute,hour,time_of_day,day,weekday,is_weekend,month
0,2025-11-28 00:00:00,1912.400000,full,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,0,Night,28,4,0,11
1,2025-11-28 00:01:00,1908.233333,full,1912.400000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,1,0,Night,28,4,0,11
2,2025-11-28 00:02:00,1643.183333,full,1908.233333,1912.400000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,2,0,Night,28,4,0,11
3,2025-11-28 00:03:00,2992.950000,full,1643.183333,1908.233333,1912.400000,NaN,NaN,NaN,NaN,...,NaN,NaN,0,3,0,Night,28,4,0,11
4,2025-11-28 00:04:00,1106.000000,full,2992.950000,1643.183333,1908.233333,1912.4,NaN,NaN,NaN,...,NaN,NaN,0,4,0,Night,28,4,0,11


In [172]:
dfs["1min"].dtypes

timestamp          datetime64[ns]
load                      float64
workday                    object
lag_1                     float64
lag_2                     float64
lag_3                     float64
lag_4                     float64
lag_5                     float64
lag_10                    float64
lag_15                    float64
lag_30                    float64
lag_60                    float64
rolling_mean_5            float64
rolling_std_5             float64
rolling_max_5             float64
rolling_min_5             float64
rolling_mean_15           float64
rolling_std_15            float64
rolling_max_15            float64
rolling_min_15            float64
rolling_mean_30           float64
rolling_std_30            float64
rolling_max_30            float64
rolling_min_30            float64
rolling_mean_60           float64
rolling_std_60            float64
rolling_max_60            float64
rolling_min_60            float64
delta_5                   float64
delta_15      

In [173]:
# save each engineered dataset
for res, df in dfs.items():
    df.to_parquet(engineered_filepath(res), index=False)

### Step 4: Model Preprocessing

In [ ]:
def preprocess(df: pd.DataFrame, scale: bool = True):
    # 1. Drop NaNs
    df = df.dropna().copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # 2. Encode categoricals
    enc = OrdinalEncoder()
    df[CATEGORICAL_COLS] = enc.fit_transform(df[CATEGORICAL_COLS])

    # 3. Dynamic split based on last N days
    max_date = df["timestamp"].max()
    test_start = max_date - pd.Timedelta(days=TEST_DAYS)
    val_start  = test_start - pd.Timedelta(days=VAL_DAYS)

    df_train = df[df["timestamp"] <= val_start]
    df_val   = df[(df["timestamp"] > val_start) & (df["timestamp"] <= test_start)]
    df_test  = df[df["timestamp"] > test_start]

    print(f"Train: {df_train.shape} | {df['timestamp'].min().date()} → {val_start.date()}")
    print(f"Val:   {df_val.shape}   | {val_start.date()} → {test_start.date()}")
    print(f"Test:  {df_test.shape}  | {test_start.date()} → {max_date.date()}")

    # 4. X / y split
    X_train, y_train = df_train.drop(columns=DROP_COLS), df_train["load"]
    X_val,   y_val   = df_val.drop(columns=DROP_COLS),   df_val["load"]
    X_test,  y_test  = df_test.drop(columns=DROP_COLS),  df_test["load"]

    # 5. Scale (fit on train only)
    if scale:
        numeric_cols = [c for c in X_train.columns if c not in CATEGORICAL_COLS]
        scaler = StandardScaler()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_val[numeric_cols]   = scaler.transform(X_val[numeric_cols])
        X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

    y_train = y_train.to_frame()
    y_val   = y_val.to_frame()
    y_test  = y_test.to_frame()

    return X_train, y_train, X_val, y_val, X_test, y_test


# ── RUN ───────────────────────────────────────────────────────────────────────
for res in RESAMPLE_RESOLUTIONS:
    print(f"\nPreprocessing {res} dataset:")
    df = pd.read_parquet(engineered_filepath(res))
    X_train, y_train, X_val, y_val, X_test, y_test = preprocess(df, scale=True)
    X_train.to_parquet(preprocess_filepath(res, "X_train"), index=False)
    y_train.to_parquet(preprocess_filepath(res, "y_train"), index=False)
    X_val.to_parquet(preprocess_filepath(res, "X_val"), index=False)
    y_val.to_parquet(preprocess_filepath(res, "y_val"), index=False)
    X_test.to_parquet(preprocess_filepath(res, "X_test"), index=False)
    y_test.to_parquet(preprocess_filepath(res, "y_test"), index=False)

### Step 5: Model Training

Train different models and save prediction results

In [196]:
dfs = {}
for res in RESAMPLE_RESOLUTIONS:
    print(res, ":", end=" ")
    X_train = pd.read_parquet(preprocess_filepath(res, "X_train"))
    X_val   = pd.read_parquet(preprocess_filepath(res, "X_val"))
    X_test  = pd.read_parquet(preprocess_filepath(res, "X_test"))
    y_train = pd.read_parquet(preprocess_filepath(res, "y_train"))
    y_val   = pd.read_parquet(preprocess_filepath(res, "y_val"))
    y_test  = pd.read_parquet(preprocess_filepath(res, "y_test"))
    dfs[res] = (X_train, y_train, X_val, y_val, X_test, y_test)
    print(f"train={X_train.shape} | val={X_val.shape} | test={X_test.shape}")


30s : train=(71222, 42) | val=(8406, 42) | test=(8640, 42)
1min : train=(35444, 42) | val=(4176, 42) | test=(4320, 42)
2min : train=(17756, 42) | val=(2029, 42) | test=(2160, 42)
5min : train=(7140, 42) | val=(802, 42) | test=(864, 42)
10min : train=(3540, 42) | val=(371, 42) | test=(432, 42)
15min : train=(2340, 42) | val=(288, 42) | test=(288, 42)


#### `Baseline Model 1`: Average load per workday

In [ ]:
def train_model(X_train, y_train, X_val, y_val):
    # Placeholder for training code
    pass

### Step 6: Model Evaluation